In [ ]:
from pathlib import Path
import json
from collections import defaultdict

import numpy as np
import pandas as pd
from scipy.stats import ks_2samp

ROOT = Path.cwd()
RESULTS_ROOT = ROOT / "number_extracted"
GT_ROOT = ROOT / "ground_truth_values"

print("ROOT:", ROOT)
print("RESULTS_ROOT exists:", RESULTS_ROOT.exists())
print("GT_ROOT exists:", GT_ROOT.exists())

In [ ]:
# Build task dictionary keyed by uid
# Each UID stores task metadata, ground truth values, and per-model extracted numbers.
tasks = {}
result_files = sorted(RESULTS_ROOT.rglob("all_results.json"))


def model_group_name(result_file, row):
    model_name = row.get("model_used")
    if model_name is None:
        return None

    temp_folder = next((part for part in result_file.parts if part.startswith("temp_")), None)
    if temp_folder is None:
        return model_name

    return f"{model_name} | {temp_folder}"


for result_file in result_files:
    with result_file.open("r", encoding="utf-8") as f:
        rows = json.load(f)

    for row in rows:
        uid = row.get("uid")
        if uid is None:
            continue

        if uid not in tasks:
            tasks[uid] = {
                "uid": uid,
                "id": row.get("id"),
                "category": row.get("category"),
                "subcategory": row.get("subcategory"),
                "prompt_title": row.get("prompt_title"),
                "ground_truth_values": ground_truth_by_uid.get(uid, []),
                "model_values": defaultdict(list),
            }

        model_name = model_group_name(result_file, row)
        extracted = row.get("extracted_number")

        if model_name is None:
            continue

        if isinstance(extracted, (int, float)) and np.isfinite(extracted):
            tasks[uid]["model_values"][model_name].append(float(extracted))

# Convert defaultdicts to plain dicts for easier inspection and export.
for uid in tasks:
    tasks[uid]["model_values"] = dict(tasks[uid]["model_values"])

print(f"Loaded {len(result_files):,} result files.")
print(f"Built task dictionary for {len(tasks):,} UIDs.")

In [ ]:
from scipy.stats import gaussian_kde
from scipy.spatial.distance import jensenshannon
from scipy.stats import wasserstein_distance, energy_distance

DISCRETE_KS_PERMUTATIONS = 10 ** 6
DISCRETE_KS_BATCH_SIZE = 20_000
DISCRETE_KS_SEED = 20240521

# Shared, fixed-seed generator: all discrete KS calls draw from the same
# stream so a full run of the notebook is reproducible end to end.
_discrete_ks_rng = np.random.default_rng(DISCRETE_KS_SEED)


def is_discrete_target(*labels):
    """Targets whose id/category live under Discrete_distributions/ use the discrete KS calibration."""
    return any(label and "Discrete_distributions" in str(label) for label in labels)


def _ks_statistic_from_counts(counts_a, counts_b, n_a, n_b):
    """Two-sample KS statistic from category counts over a shared, ordered support."""
    cdf_a = np.cumsum(counts_a, axis=-1) / n_a
    cdf_b = np.cumsum(counts_b, axis=-1) / n_b
    return np.max(np.abs(cdf_a - cdf_b), axis=-1)


def ks_score_discrete(a, b, n_permutations=DISCRETE_KS_PERMUTATIONS, batch_size=DISCRETE_KS_BATCH_SIZE, rng=None):
    """
    Two-sample KS test calibrated for discrete targets.

    Keeps the standard two-sample KS statistic but calibrates its p-value by
    Monte Carlo permutation: each permutation draws a multivariate
    hypergeometric allocation of the pooled sample into a group of size n_a
    (the remainder forms the group of size n_b). This is exact under ties
    and does not rely on the continuous-null assumption.

    Uses B = 10**6 permutations by default, with p = (b + 1) / (B + 1) so the
    estimate never hits exactly zero.
    """
    rng = _discrete_ks_rng if rng is None else rng
    a = np.asarray(a, dtype=float).ravel()
    b = np.asarray(b, dtype=float).ravel()
    n_a, n_b = a.size, b.size

    if n_a < 1 or n_b < 1:
        return {"ks_statistic": float("nan"), "ks_p_value": float("nan")}

    support, inverse = np.unique(np.concatenate([a, b]), return_inverse=True)
    pooled_counts = np.bincount(inverse, minlength=support.size).astype(np.int64)
    counts_a_obs = np.bincount(inverse[:n_a], minlength=support.size).astype(np.int64)
    counts_b_obs = pooled_counts - counts_a_obs

    stat_obs = float(_ks_statistic_from_counts(counts_a_obs, counts_b_obs, n_a, n_b))

    n_total = int(pooled_counts.sum())
    if n_a == n_total or n_b == n_total:
        # Degenerate split: no randomness possible, statistic is always 0.
        return {"ks_statistic": stat_obs, "ks_p_value": 1.0}

    B = int(n_permutations)
    ge_count = 0
    remaining = B
    while remaining > 0:
        cur = min(batch_size, remaining)
        counts_a_perm = rng.multivariate_hypergeometric(pooled_counts, n_a, size=cur)
        counts_b_perm = pooled_counts[None, :] - counts_a_perm
        stat_perm = _ks_statistic_from_counts(counts_a_perm, counts_b_perm, n_a, n_b)
        ge_count += int(np.sum(stat_perm >= stat_obs))
        remaining -= cur

    p_value = (ge_count + 1) / (B + 1)
    return {"ks_statistic": stat_obs, "ks_p_value": float(p_value)}


def ks_score_continuous(a, b):
    """Two-sample KS test for continuous targets, using the exact null distribution."""
    stat, p_value = ks_2samp(a, b, method="exact")
    return {
        "ks_statistic": float(stat),
        "ks_p_value": float(p_value),
    }


def ks_score(a, b, is_discrete=False):
    """
    Dispatches to the discrete- or continuous-calibrated KS test.

    Discrete targets (id/category under Discrete_distributions/) are scored
    with a Monte Carlo permutation-calibrated p-value (see
    ks_score_discrete), since the continuous-null KS p-value does not apply
    under heavy ties. Continuous targets (and everything else) use the
    two-sample KS test with the exact null distribution (method="exact").
    """
    if is_discrete:
        return ks_score_discrete(a, b)
    return ks_score_continuous(a, b)


def _w1_sorted_equal(x_sorted, y_sorted):
    """Wasserstein-1 between two equal-size 1D samples, both pre-sorted."""
    return np.mean(np.abs(x_sorted - y_sorted))


def _w1_sorted(x, y):
    """Wasserstein-1 for 1D samples of possibly unequal size. Sorts internally."""
    x = np.sort(x)
    y = np.sort(y)
    if x.size == y.size:
        return np.mean(np.abs(x - y))
    # Fallback: scipy handles unequal sizes via the CDF-integral form
    return wasserstein_distance(x, y)


def distance_distribution_scores(a, b, n_resamples=999, rng=20240521):
    """Debiased Wasserstein-1 and energy distance with a shared permutation null."""
    rng = np.random.default_rng(rng)
    a = np.asarray(a, dtype=float).ravel()
    b = np.asarray(b, dtype=float).ravel()

    n_a = a.size
    n_b = b.size
    if n_a < 2 or n_b < 2:
        return {k: float("nan") for k in (
            "wasserstein_debiased", "wasserstein_z",
            "energy_debiased", "energy_z",
        )}

    pooled = np.concatenate([a, b])
    n_total = pooled.size
    equal_sizes = (n_a == n_b)

    # Observed statistics
    if equal_sizes:
        w_obs = _w1_sorted_equal(np.sort(a), np.sort(b))
    else:
        w_obs = _w1_sorted(a, b)
    e_obs = float(energy_distance(a, b))

    # Shared permutation null
    w_null = np.empty(n_resamples)
    e_null = np.empty(n_resamples)
    for i in range(n_resamples):
        perm = rng.permutation(n_total)
        x = pooled[perm[:n_a]]
        y = pooled[perm[n_a:]]
        if equal_sizes:
            w_null[i] = _w1_sorted_equal(np.sort(x), np.sort(y))
        else:
            w_null[i] = _w1_sorted(x, y)
        e_null[i] = energy_distance(x, y)

    def _summarize(obs, null):
        mean = null.mean()
        std = null.std()
        debiased = float(obs - mean)
        if std < 1e-12:
            z = float("nan")
        else:
            z = float((obs - mean) / std)
        return debiased, z

    w_debiased, w_z = _summarize(w_obs, w_null)
    e_debiased, e_z = _summarize(e_obs, e_null)

    return {
        "wasserstein_debiased": w_debiased,
        "wasserstein_z": w_z,
        "energy_debiased": e_debiased,
        "energy_z": e_z,
    }


def js_divergence_score(a, b, grid_size=512, pad=0.1):
    """Jensen-Shannon divergence via KDE on a shared grid."""
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)

    if a.size < 2 or b.size < 2:
        return np.nan
    if np.allclose(a.min(), a.max()) and np.allclose(b.min(), b.max()):
        return 0.0 if np.isclose(a[0], b[0]) else np.nan

    lo = min(a.min(), b.min())
    hi = max(a.max(), b.max())
    span = hi - lo
    lo -= pad * span
    hi += pad * span
    grid = np.linspace(lo, hi, grid_size)

    try:
        p = gaussian_kde(a)(grid)
        q = gaussian_kde(b)(grid)
    except (np.linalg.LinAlgError, ValueError):
        return np.nan

    p_sum, q_sum = p.sum(), q.sum()
    if p_sum == 0 or q_sum == 0:
        return np.nan
    p /= p_sum
    q /= q_sum

    js_distance = jensenshannon(p, q)
    return float(js_distance ** 2)


RANDOM_MODEL_NAME = "random/last-100-gt"


def compute_metrics_records(gt_start, gt_end, include_random_baseline=True):
    records = []

    for uid, task in tasks.items():
        gt_full = ground_truth_full_by_uid.get(uid, [])
        gt_slice = gt_full[gt_start:gt_end]
        if len(gt_slice) == 0:
            continue

        gt_arr = np.asarray(gt_slice, dtype=float)
        is_discrete = is_discrete_target(task.get("id"), task.get("category"))

        for model_name, model_vals in task.get("model_values", {}).items():
            if len(model_vals) == 0:
                continue

            pred_arr = np.asarray(model_vals, dtype=float)
            ks = ks_score(gt_arr, pred_arr, is_discrete=is_discrete)

            rec = {
                "uid": uid,
                "id": task.get("id"),
                "category": task.get("category"),
                "subcategory": task.get("subcategory"),
                "prompt_title": task.get("prompt_title"),
                "model": model_name,
                "ks_statistic": ks["ks_statistic"],
                "ks_p_value": ks["ks_p_value"],
                "js_divergence": js_divergence_score(gt_arr, pred_arr),
            }

            for _n in [1, 2, 5, 10, 20, 50, 100]:
                ks_at_n = ks_score(gt_arr, pred_arr[:_n], is_discrete=is_discrete)
                rec.update( {'ks_{}_statistic'.format(_n):ks_at_n["ks_statistic"],
                            'ks_{}_p_value'.format(_n):ks_at_n["ks_p_value"]})
            
            records.append(rec)

        if include_random_baseline and len(gt_full) >= 100:
            random_pred = np.asarray(gt_full[-100:], dtype=float)
            ks = ks_score(gt_arr, random_pred, is_discrete=is_discrete)
            
            rec = {
                "uid": uid,
                "id": task.get("id"),
                "category": task.get("category"),
                "subcategory": task.get("subcategory"),
                "prompt_title": task.get("prompt_title"),
                "model": RANDOM_MODEL_NAME,
                "ks_statistic": ks["ks_statistic"],
                "ks_p_value": ks["ks_p_value"],
                "js_divergence": js_divergence_score(gt_arr, random_pred),
            }

            for _n in [1, 2, 5, 10, 20, 50, 100]:
                ks_at_n = ks_score(gt_arr, random_pred[:_n], is_discrete=is_discrete)
                rec.update( {'ks_{}_statistic'.format(_n):ks_at_n["ks_statistic"],
                            'ks_{}_p_value'.format(_n):ks_at_n["ks_p_value"]})
            
            records.append(rec)

    return records


metrics_df_full = pd.DataFrame(compute_metrics_records(0, 10000, include_random_baseline=True))

In [ ]:
threshold = 0.0001

metrics_df_full_first_1000['acc_ks'] = metrics_df_full_first_1000['ks_p_value'] > threshold

for _n in [1, 2, 5, 10, 20, 50, 100]:
    metrics_df_full_first_1000['acc_ks_{}'.format(_n)] = metrics_df_full_first_1000['ks_{}_p_value'.format(_n)] > threshold


df_accs = metrics_df_full_first_1000[['acc_ks_50', 'acc_ks_100', 'model']].groupby('model').mean()

sorted_df_accs = df_accs.sort_values('model', ascending=False)
sorted_df_accs

In [ ]:
'''

Get accuracy df

'''
threshold = 0.0001

metrics_df_full['acc_ks'] = metrics_df_full['ks_p_value'] > threshold

for _n in [1, 2, 5, 10, 20, 50, 100]:
    metrics_df_full['acc_ks_{}'.format(_n)] = metrics_df_full['ks_{}_p_value'.format(_n)] > threshold


df_accs = metrics_df_full[['acc_ks', 'acc_ks_1', 'acc_ks_2', 'acc_ks_5', 'acc_ks_10', 'acc_ks_20', 'acc_ks_50', 'acc_ks_100', 'model']].groupby('model').mean()

sorted_df_accs = df_accs.sort_values('acc_ks', ascending=False)
sorted_df_accs